## --- THE LOGIC (Same as the main script) ---

In [5]:
import requests
import pandas as pd

def format_steam_price(game_data, price_key):
    # Safely handles 'is_free' games and missing price_overview
    # Returns a formatted string like '19.99 USD' or '0.00 FREE'
    if game_data.get('is_free') is True:
        return "0.00 FREE"
    
    p_ov = game_data.get('price_overview')
    if not p_ov:
        return None
        
    price_raw = p_ov.get(price_key)
    currency = p_ov.get('currency', 'USD')
    
    if price_raw is None:
        return f"0.00 {currency}"
        
    p_str = str(price_raw)
    # Handle edge cases where price might be less than 3 digits (e.g., 50 cents)
    if len(p_str) <= 2:
        return f"0.{p_str.zfill(2)} {currency}"
    
    return f"{p_str[:-2]}.{p_str[-2:]} {currency}"

def test_single_game(appid):
    url = f"https://store.steampowered.com/api/appdetails?appids={appid}&cc=de&l=english"
    print(f"Testing AppID: {appid}...")
    
    resp = requests.get(url)
    data_json = resp.json()
    
    if data_json and data_json[str(appid)]['success']:
        g = data_json[str(appid)]['data']
        
        # Extracting lists/dicts safely
        devs = g.get('developers', [])
        pubs = g.get('publishers', [])
        cats = g.get('categories', [])
        gens = g.get('genres', [])
        ss   = g.get('screenshots', [])
        recs = g.get('recommendations', {})
        
        results = {
            "NAME": g.get('name'),
            "IS_FREE": g.get('is_free'),
            "DETAILED_DESCRIPTION": g.get('detailed_description'),
            "SHORT_DESCRIPTION": g.get('short_description'),
            "HEADER_IMAGE": g.get('header_image'),
            "WEBSITE": g.get('website'),
            "DEVELOPERS": ", ".join(devs) if devs else None,
            "PUBLISHERS": ", ".join(pubs) if pubs else None,
            "INITIAL_PRICE": format_steam_price(g, 'initial'),
            "FINAL_PRICE": format_steam_price(g, 'final'),
            "CATEGORIES": ", ".join([c['description'] for c in cats]) if cats else None,
            "GENRES": ", ".join([gen['description'] for gen in gens]) if gens else None,
            "SS1": ss[0]['path_full'] if len(ss) > 0 else None,
            "SS2": ss[1]['path_full'] if len(ss) > 1 else None,
            "SS3": ss[2]['path_full'] if len(ss) > 2 else None,
            "RECOMMENDATIONS": recs.get('total'),
            "RELEASE_DATE": g.get('release_date', {}).get('date'),
            "BACKGROUND_IMAGE": g.get('background')
        }
        return results
    else:
        return {"ERROR": "Success is False (could be a DLC, Bundle, or Restricted)"}


#### --- THE TEST RUNS ---

#### 1. Elden Ring (Paid Game)
#### 2. Dota 2 (Free to Play - No price_overview)
#### 3. Portal 2 (Classic - Usually has discounts)
#### 4. A random high ID (Modern Indie)
#### 5. Terraria
#### 6. Trove
#### 7. Sekiro
#### 8. Don't Starve Together

In [6]:
test_ids = [1245620, 570, 620, 1942280, 105600, 304050, 814380, 322330, 1627720]

final_results = []
for tid in test_ids:
    res = test_single_game(tid)
    res['APPID'] = tid
    final_results.append(res)

# Display as a Table for easy comparison
pd.set_option('display.max_colwidth', None)
df = pd.DataFrame(final_results)
df = df[['APPID', 'NAME', 'IS_FREE', 'SHORT_DESCRIPTION', 
         'HEADER_IMAGE', 'WEBSITE', 'DEVELOPERS', 'PUBLISHERS', 
         'INITIAL_PRICE', 'FINAL_PRICE', 'CATEGORIES', 'GENRES', 
         'SS1', 'SS2', 'SS3', 'RECOMMENDATIONS', 'RELEASE_DATE', 'BACKGROUND_IMAGE']]
display(df)

Testing AppID: 1245620...
Testing AppID: 570...
Testing AppID: 620...
Testing AppID: 1942280...
Testing AppID: 105600...
Testing AppID: 304050...
Testing AppID: 814380...
Testing AppID: 322330...
Testing AppID: 1627720...


,APPID,NAME,IS_FREE,SHORT_DESCRIPTION,HEADER_IMAGE,WEBSITE,DEVELOPERS,PUBLISHERS,INITIAL_PRICE,FINAL_PRICE,CATEGORIES,GENRES,SS1,SS2,SS3,RECOMMENDATIONS,RELEASE_DATE,BACKGROUND_IMAGE
0,1245620,ELDEN RING,False,"THE CRITICALLY ACCLAIMED FANTASY ACTION RPG. Rise, Tarnished, and be guided by grace to brandish the power of the Elden Ring and become an Elden Lord in the Lands Between.",https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/1245620/header.jpg?t=1767883716,NaN,"FromSoftware, Inc.","FromSoftware, Inc., Bandai Namco Entertainment",59.99 EUR,38.99 EUR,"Single-player, Multi-player, PvP, Online PvP, Co-op, Online Co-op, Steam Achievements, Full controller support, Steam Trading Cards, Camera Comfort, Custom Volume Controls, Playable without Timed Input, Save Anytime, Stereo Sound, Surround Sound, Steam Cloud, Family Sharing","Action, RPG",https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/1245620/ss_943bf6fe62352757d9070c1d33e50b92fe8539f1.1920x1080.jpg?t=1767883716,https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/1245620/ss_dcdac9e4b26ac0ee5248bfd2967d764fd00cdb42.1920x1080.jpg?t=1767883716,https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/1245620/ss_3c41384a24d86dddd58a8f61db77f9dc0bfda8b5.1920x1080.jpg?t=1767883716,808977,"24 Feb, 2022",https://store.akamai.steamstatic.com/images/storepagebackground/app/1245620?t=1767883716
1,570,Dota 2,True,"Every day, millions of players worldwide enter battle as one of over a hundred Dota heroes. And no matter if it's their 10th hour of play or 1,000th, there's always something new to discover. With regular updates that ensure a constant evolution of gameplay, features, and heroes, Dota 2 has taken on a life of its own.",https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/570/header.jpg?t=1769535998,http://www.dota2.com/,Valve,Valve,0.00 FREE,0.00 FREE,"Multi-player, Co-op, Steam Trading Cards, Steam Workshop, SteamVR Collectibles, In-App Purchases, Camera Comfort, Color Alternatives, Custom Volume Controls, Valve Anti-Cheat enabled, Steam Timeline","Action, Strategy, Free To Play",https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/570/ss_ad8eee787704745ccdecdfde3a5cd2733704898d.1920x1080.jpg?t=1769535998,https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/570/ss_7ab506679d42bfc0c0e40639887176494e0466d9.1920x1080.jpg?t=1769535998,https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/570/ss_c9118375a2400278590f29a3537769c986ef6e39.1920x1080.jpg?t=1769535998,14365,"9 Jul, 2013",https://store.akamai.steamstatic.com/images/storepagebackground/app/570?t=1769535998
2,620,Portal 2,False,The &quot;Perpetual Testing Initiative&quot; has been expanded to allow you to design co-op puzzles for you and your friends!,https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/620/header.jpg?t=1745363004,http://www.thinkwithportals.com/,Valve,Valve,9.75 EUR,1.95 EUR,"Single-player, Multi-player, Co-op, Online Co-op, Shared/Split Screen Co-op, Shared/Split Screen, Steam Achievements, Full controller support, Steam Trading Cards, Captions available, Steam Workshop, Steam Workshop, Camera Comfort, Custom Volume Controls, Playable without Timed Input, Save Anytime, Stereo Sound, Surround Sound, Steam Cloud, Stats, Includes level editor, Commentary available, Remote Play on Phone, Remote Play on Tablet, Remote Play on TV, Remote Play Together, Family Sharing","Action, Adventure",https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/620/ss_f3f6787d74739d3b2ec8a484b5c994b3d31ef325.1920x1080.jpg?t=1745363004,https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/620/ss_6a4f5afdaa98402de9cf0b59fed27bab3256a6f4.1920x1080.jpg?t=1745363004,https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/620/ss_0cdd90fafc160b52d08b303d205f9fd4e83cf164.1920x1080.jpg?t=1745363004,380304,"18 Apr, 2011",https://store.akamai.steamstatic.com/images/storepagebackg